In [ ]:
!git clone https://github.com/lanadanolic/custom-hybrid-cnn-for-cifar10.git

In [ ]:
%cd /content/custom-hybrid-cnn-for-cifar10

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from sklearn.model_selection import train_test_split

In [ ]:
print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
PROJECT_DIR = Path.cwd()

if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

print("Project directory:", PROJECT_DIR)

In [ ]:
from src.models import (
    build_standard_cnn,
    build_depthwise_cnn,
    build_dilated_cnn,
    build_hybrid_cnn
)

from src.training import (
    compile_model,
    train_model
)

from src.evaluation import (
    plot_history,
    evaluate_model
)

In [ ]:
(x_train_original, y_train_original), (x_test, y_test) = (
    keras.datasets.cifar10.load_data()
)

y_train_original = y_train_original.squeeze()
y_test = y_test.squeeze()

In [ ]:
class_names = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]

In [ ]:
SEED = 42

x_train, x_val, y_train, y_val = train_test_split(
    x_train_original,
    y_train_original,
    test_size=5000,
    random_state=SEED,
    stratify=y_train_original
)

In [ ]:
print("Train:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_val = x_val.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

In [ ]:
standard_model = build_standard_cnn()

standard_model.summary()

compile_model(standard_model)

standard_history, standard_training_time = train_model(
    standard_model,
    "StandardCNN",
    x_train,
    y_train,
    x_val,
    y_val
)

In [ ]:
plot_history(
    standard_history,
    "StandardCNN"
)

standard_results = evaluate_model(
    standard_model,
    "StandardCNN",
    standard_training_time,
    x_test,
    y_test,
    class_names
)

In [ ]:
depthwise_model = build_depthwise_cnn()

depthwise_model.summary()

compile_model(depthwise_model)

depthwise_history, depthwise_training_time = train_model(
    depthwise_model,
    "DepthwiseCNN",
    x_train,
    y_train,
    x_val,
    y_val
)

In [ ]:
plot_history(
    depthwise_history,
    "DepthwiseCNN"
)

depthwise_results = evaluate_model(
    depthwise_model,
    "DepthwiseCNN",
    depthwise_training_time,
    x_test,
    y_test,
    class_names
)

In [ ]:
dilated_model = build_dilated_cnn()

dilated_model.summary()

compile_model(dilated_model)

dilated_history, dilated_training_time = train_model(
    dilated_model,
    "DilatedCNN",
    x_train,
    y_train,
    x_val,
    y_val
)


In [ ]:
plot_history(
    dilated_history,
    "DilatedCNN"
)

dilated_results = evaluate_model(
    dilated_model,
    "DilatedCNN",
    dilated_training_time,
    x_test,
    y_test,
    class_names
)

In [ ]:
hybrid_model = build_hybrid_cnn(
    dilation_rate=2
)

hybrid_model.summary()

compile_model(hybrid_model)

hybrid_history, hybrid_training_time = train_model(
    hybrid_model,
    "HybridCNN",
    x_train,
    y_train,
    x_val,
    y_val
)

In [ ]:
plot_history(
    hybrid_history,
    "HybridCNN"
)

hybrid_results = evaluate_model(
    hybrid_model,
    "HybridCNN",
    hybrid_training_time,
    x_test,
    y_test,
    class_names
)

In [ ]:
results_df = pd.DataFrame([
    standard_results,
    depthwise_results,
    dilated_results,
    hybrid_results
])

results_df

In [ ]:
# comparison of classification performance

performance_df = results_df.set_index("Model")[
    ["Accuracy", "F1-score"]
]

performance_df.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("Accuracy and F1-score comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# comparison of model size

results_df.plot(
    x="Model",
    y="Parameters",
    kind="bar",
    figsize=(8, 5),
    legend=False
)

plt.title("Number of parameters")
plt.ylabel("Parameters")
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# comparison of training time

results_df.plot(
    x="Model",
    y="Training time (s)",
    kind="bar",
    figsize=(8, 5),
    legend=False
)

plt.title("Training time")
plt.ylabel("Time (s)")
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# comparison of prediction time

results_df.plot(
    x="Model",
    y="Prediction time per image (ms)",
    kind="bar",
    figsize=(8, 5),
    legend=False
)

plt.title("Prediction time per image")
plt.ylabel("Time (ms)")
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.tight_layout()
plt.show()